In [19]:
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, ModelProvider, Model
from agents.tracing import set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel
from typing import Literal
from pydantic import BaseModel, Field

set_tracing_disabled(True)

In [20]:
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",  # Ollama's OpenAI-compatible endpoint
    api_key="ollama",  # Dummy key — Ollama doesn't need one
)

# Step 3: Wrap it in the SDK's model class
local_model = OpenAIChatCompletionsModel(
    model="minimax-m2:cloud",  # Must match what you pulled with `ollama pull`
    openai_client=ollama_client,
)

# Step 4: Create agent with local model
agent = Agent(
    name="Local Assistant",
    instructions="You are a helpful assistant running locally via Ollama. Be concise.",
    model=local_model,  # <-- This is the key line!
)

# Step 5: Run it
result = await Runner.run(agent, "What is the Agents SDK in one paragraph?")
print(result.final_output)

The OpenAI Agents SDK is a lightweight Python framework (released December 2024) for building, debugging, and deploying AI agents. It provides three core primitives: **Agents** (with instructions, tools, and handoffs), **Guardrails** (for input/output validation), and **Handoffs** (for transferring control between agents). It's designed for multi-agent workflows where you want explicit control over agent orchestration, rather than relying on implicit reasoning chains.


In [21]:
class OllamaProvider(ModelProvider):
    """Custom ModelProvider that routes all model requests to Ollama."""

    def __init__(self, model_name: str = "minimax-m2:cloud"):
        self.model_name = model_name
        self.client = AsyncOpenAI(
            base_url="http://localhost:11434/v1", api_key="ollama"
        )

    def get_model(self, model_name: str | None = None) -> Model:
        return OpenAIChatCompletionsModel(
            model=model_name or self.model_name, openai_client=self.client
        )


# Now create agents WITHOUT specifying model each time
ollama = OllamaProvider()

agent = Agent(
    name="Local Agent",
    instructions="You are a local AI assistant. Be helpful and concise.",
    model=ollama.get_model(),  # Clean and reusable!
)

result = await Runner.run(agent, "Explain what Ollama is in 2 sentences.")
print(result.final_output)

Ollama is a platform that lets you run large language models (LLMs) locally on your own computer. It makes it easy to set up and use AI models without relying on cloud services.


In [22]:
agent = Agent(
    name="LiteLLM Agent",
    instructions="You are a helpful assistant.",
    # IMPORTANT: prepend 'ollama_chat/' — tells LiteLLM to use Ollama's chat API
    model=LitellmModel(
        model="ollama/minimax-m2:cloud", base_url="http://localhost:11434"
    ),
)

result = await Runner.run(agent, "Hello! What model are you?")
print(result.final_output)

Hello! I am MiniMax-M2.1 (also known as Ernie 4.5), an AI assistant built by MiniMax. I'm here to help you with any questions or tasks you might have. How can I assist you today?


In [23]:
def get_ollama_model(
    model_name: str = "minimax-m2:cloud", base_url: str = "http://localhost:11434/v1"
):
    """Create an Ollama-backed model for the Agents SDK.

    Args:
        model_name: Ollama model name (must be pulled first)
        base_url: Ollama server URL

    Returns:
        OpenAIChatCompletionsModel ready to use with Agent(model=...)
    """
    client = AsyncOpenAI(base_url=base_url, api_key="ollama")
    return OpenAIChatCompletionsModel(model=model_name, openai_client=client)


# Usage:
# from ollama_setup import get_ollama_model
# agent = Agent(name="My Agent", instructions="...", model=get_ollama_model())
# agent = Agent(name="My Agent", instructions="...", model=get_ollama_model("llama3.1:8b"))

print("ollama_setup module ready")

ollama_setup module ready


In [26]:
local_model = OpenAIChatCompletionsModel(
    model="qwen2.5-coder:3b",
    openai_client=AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama"),
)


class ResumeAnalysis(BaseModel):
    name: str = Field(description="Candidate name")
    years_experience: int = Field(description="Total years of experience")
    top_skills: list[str] = Field(description="Top 3-5 relevant skills")
    fit_score: int = Field(description="Job fit score 1-100")
    fit_level: Literal["strong", "moderate", "weak"] = Field(description="Overall fit")
    summary: str = Field(description="2-sentence summary for hiring manager")


resume_agent = Agent(
    name="Resume Analyzer",
    instructions="""
    You are an HR AI assistant that analyzes resumes against job requirements.
    
    JOB: Senior Python Developer
    REQUIREMENTS: 5+ years Python, FastAPI/Django, SQL, cloud experience (AWS/GCP)
    NICE TO HAVE: AI/ML, Docker, Kubernetes, system design
    
    Analyze the resume text and score the candidate's fit honestly.
    """,
    model=local_model,
    output_type=ResumeAnalysis,
)

# Simulate a resume
resume_text = """
Ahmed Hassan — Lahore, Pakistan
7 years experience in software development.
Skills: Python, FastAPI, PostgreSQL, Redis, Docker, AWS (EC2, Lambda, S3)
Previous: Lead Developer at SAI Lab (3 years), Senior Dev at SAI (4 years)
Projects: Built a real-time analytics pipeline processing 1M events/day.
Education: BS Computer Science, UAF
"""

result = await Runner.run(resume_agent, resume_text)
r = result.final_output

print(f"{r.name}")
print(f"Experience: {r.years_experience} years")
print(f"Skills: {', '.join(r.top_skills)}")
print(f"Fit Score: {r.fit_score}/100 ({r.fit_level})")
print(f"Summary: {r.summary}")

Ahmed Hassan
Experience: 7 years
Skills: FastAPI, PostgreSQL, Docker, AWS
Fit Score: 0/100 (moderate)
Summary: Strong Python developer with experience in FastAPI/Django, SQL (PostgreSQL), and cloud experience (AWS). Previous leadership roles at both SAI Lab and SAI with projects involving real-time analytics pipelines processing 1M events per day.
